# 02 — Lag Verification (Cross-correlation vs Reference)

Fine-tune per-file time offsets by cross-correlating each instrument's CH4_ppm against
a trusted reference. Produces JSON offset files consumed by `clean.py -o` in Phase 4.

**Pipeline position:** `ts_corrected/` → `cleaned/` → **lag JSONs** → `recleaned/` → `merged/`

## Cross-correlation strategy
| Section | Instrument | Reference | Dates |
|---|---|---|---|
| A | Ultra 460 | Picarro | Feb 3–12 (WYO) |
| B | Ultra 321 | Picarro | WYO dates; MML dates (Feb 4, Mar 8–10) auto-suggest 0s |
| C | Pico 017 (WYO) | Picarro | Jan 19–22, Feb 5–12 |
| D | Pico 017 (MML) + LGR | Ultra 321 | Feb 4, Mar 8–10 |

## Output JSONs (saved to `mobile-slv/offsets/`)
- `ultra460_lag.json` + `ultra460_rejected.json`
- `ultra321_lag.json` + `ultra321_rejected.json`
- `pico017_lag.json` + `pico017_rejected.json`
- `lgr_lag.json` + `lgr_rejected.json`

Keys are `*.txt` filenames (raw instrument files) — the format `clean.py -o` expects.

## Workflow
1. Run Imports, Config, Helpers, and Load Picarro cells once
2. For each section: run Auto-correlate, then the Widget cell for visual review
3. In the widget: **Commit & Next** (good), **Mark Bad & Next** (exclude), **Next** (skip — auto lag used at save)
4. Run the Save cell when done reviewing

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.widgets as mwidgets
from pathlib import Path

%matplotlib widget

## Config

In [ ]:
CLEANED_DIR  = Path('/uufs/chpc.utah.edu/common/home/lin-group24/agm/Mobile_SLV/Data/2026/cleaned')
OFFSETS_OUT  = Path('/uufs/chpc.utah.edu/common/home/u0890904/LAIR_1/Projects/Mobile_SLV/Code/mobile-slv/offsets')

PICARRO_DIR  = CLEANED_DIR / 'WYO_picarro'
ULTRA460_DIR = CLEANED_DIR / 'WYO_aerisultra460' / 'Raw'
ULTRA321_DIR = CLEANED_DIR / 'LANL_aerisultra321' / 'Raw'
PICO017_DIR  = CLEANED_DIR / 'LANL_aerispico017'  / 'Raw'
LGR_DIR      = CLEANED_DIR / 'UOU_LGR'

CH4_COL    = 'CH4_ppm'
MAX_LAG_S  = 600
RESAMPLE_S = 1

# MML-only deployment dates (Toughbook, no Picarro co-location)
# Everything before Feb 3 was Toughbook/MML; RPi files for those dates are accidental duplicates
MML_DATE_TAGS = {'260119', '260120', '260121', '260122', '260202', '260204', '260308', '260310'}

## Helpers — data

In [ ]:
def load_cleaned(path, col=CH4_COL):
    df = pd.read_csv(path, index_col='TIMESTAMP', parse_dates=True)
    if col not in df.columns:
        raise KeyError(f'{col!r} not found in {path.name}. Available: {df.columns.tolist()}')
    return df[col].dropna()

def resample_series(s, freq_s=RESAMPLE_S):
    s = s.resample(f'{freq_s}s').mean()
    return s.interpolate(method='time', limit=10)

def load_all_ref(directory, col=CH4_COL):
    files = sorted(directory.glob('*_clean.csv'))
    if not files:
        raise FileNotFoundError(f'No *_clean.csv files found in {directory}')
    print(f'  Loading {len(files)} file(s) from {directory.name}...')
    combined = pd.concat([load_cleaned(f, col) for f in files]).sort_index()
    combined = combined[~combined.index.duplicated(keep='first')]
    return resample_series(combined)

def load_ref_from_files(file_list, col=CH4_COL):
    combined = pd.concat([load_cleaned(f, col) for f in file_list]).sort_index()
    combined = combined[~combined.index.duplicated(keep='first')]
    return resample_series(combined)

def cross_correlate(ref, sig, max_lag_s=MAX_LAG_S, freq_s=RESAMPLE_S):
    start = max(ref.index[0], sig.index[0])
    end   = min(ref.index[-1], sig.index[-1])
    if start >= end:
        print('    [warn] No overlapping time window — defaulting to 0s')
        return 0.0
    combined = pd.DataFrame({'r': ref[start:end], 's': sig[start:end]}).dropna()
    if len(combined) < 10:
        print(f'    [warn] Only {len(combined)} overlapping point(s) — defaulting to 0s')
        return 0.0
    r_arr    = (combined['r'] - combined['r'].mean()).values
    s_arr    = (combined['s'] - combined['s'].mean()).values
    max_samp = max_lag_s // freq_s
    corr     = np.correlate(r_arr, s_arr, mode='full')
    lags     = np.arange(-(len(r_arr) - 1), len(r_arr))
    mask     = np.abs(lags) <= max_samp
    return float(lags[mask][np.argmax(corr[mask])] * freq_s)

def json_key(cleaned_path):
    """Map *_clean.csv -> *.txt — matches how clean.py -o looks up per-file offsets."""
    return cleaned_path.name.replace('_clean.csv', '.txt')

def is_mml(path):
    return path.stem.split('_')[1] in MML_DATE_TAGS

def load_json(path):
    if not path.exists():
        return {}
    with open(path) as f:
        return json.load(f)

def save_json(data, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)
    print(f'  Saved -> {path}')

def auto_correlate(test_files, ref_data):
    """Cross-correlate each file against ref_data; print table; return suggestions dict."""
    suggestions = {}
    hdr = '{:>4}  {:<52}  {:>10}'.format('IDX', 'FILE', 'AUTO LAG')
    print(hdr)
    print('-' * 72)
    for i, f in enumerate(test_files):
        key = json_key(f)
        sig = resample_series(load_cleaned(f))
        lag = cross_correlate(ref_data, sig)
        suggestions[key] = lag
        print(f'[{i:>2}]  {f.name:<52}  {lag:>+10.0f}s')
    return suggestions

def save_offsets(confirmed, rejected, stem):
    """Merge confirmed lags into existing JSON; write rejected list."""
    offset_path   = OFFSETS_OUT / f'{stem}_lag.json'
    rejected_path = OFFSETS_OUT / f'{stem}_rejected.json'
    clean_conf = {k: v for k, v in confirmed.items() if k not in rejected}
    existing   = load_json(offset_path)
    existing   = {k: v for k, v in existing.items() if k not in rejected}
    existing.update(clean_conf)
    save_json(existing, offset_path)
    save_json(sorted(rejected), rejected_path)
    print(f'\n  Confirmed : {len(clean_conf)}  |  Rejected : {len(rejected)}')
    for k, v in existing.items():
        print(f'    {k:<54} {v:>+6.1f}s')
    if rejected:
        print('\n  Rejected:')
        for k in sorted(rejected):
            print(f'    {k}')

print('Data helpers loaded.')

## Helpers — interactive review widget

In [ ]:
def make_review_widget(ref_data, test_files, test_name, suggestions, confirmed, rejected):
    """
    Interactive review widget. Modifies `confirmed` and `rejected` in-place.
    Commit & Next = save lag | Mark Bad & Next = exclude | Next = skip.
    """
    state = {'idx': 0, 'ref_win': None, 'test_win': None}

    plt.close('all')
    fig = plt.figure(figsize=(15, 6))
    ax  = fig.add_axes([0.07, 0.40, 0.91, 0.53])

    ax_lag   = fig.add_axes([0.07, 0.28, 0.87, 0.03])
    ax_scale = fig.add_axes([0.07, 0.21, 0.87, 0.03])
    ax_voff  = fig.add_axes([0.07, 0.14, 0.87, 0.03])

    s_lag   = mwidgets.Slider(ax_lag,   'Lag (s)',  -MAX_LAG_S, MAX_LAG_S, valinit=0, valstep=1)
    s_scale = mwidgets.Slider(ax_scale, 'V-Scale',  0.1, 5.0, valinit=1.0)
    s_voff  = mwidgets.Slider(ax_voff,  'V-Offset', -500, 500, valinit=0.0)

    ax_prev   = fig.add_axes([0.07, 0.04, 0.11, 0.07])
    ax_bad    = fig.add_axes([0.21, 0.04, 0.16, 0.07])
    ax_commit = fig.add_axes([0.42, 0.04, 0.20, 0.07])
    ax_next   = fig.add_axes([0.82, 0.04, 0.11, 0.07])

    btn_prev   = mwidgets.Button(ax_prev,   '<- Prev',         color='0.85')
    btn_bad    = mwidgets.Button(ax_bad,    'Mark Bad & Next', color='#fca5a5')
    btn_commit = mwidgets.Button(ax_commit, 'Commit & Next',   color='#bbf7d0')
    btn_next   = mwidgets.Button(ax_next,   'Next ->',         color='0.85')

    line_ref  = [None]
    line_test = [None]

    def _status(key):
        if key in rejected: return 'BAD'
        if key in confirmed: return 'OK'
        return '...'

    def _title():
        f     = test_files[state['idx']]
        key   = json_key(f)
        lag   = s_lag.val
        total = len(test_files)
        done  = len(confirmed) + len(rejected)
        idx   = state['idx']
        return (f'[{_status(key)}]  [{idx}/{total-1}]  {f.name}'
                f'  |  lag={lag:+.0f}s  ({done}/{total} reviewed)')

    def load_and_draw(idx):
        f   = test_files[idx]
        key = json_key(f)
        lag = confirmed.get(key, suggestions.get(key, 0.0))

        test_rs = resample_series(load_cleaned(f))
        t_start = max(ref_data.index[0], test_rs.index[0])
        t_end   = min(ref_data.index[-1], test_rs.index[-1])
        has_overlap = t_start < t_end
        if has_overlap:
            state['ref_win']  = ref_data[t_start:t_end]
            state['test_win'] = test_rs[t_start:t_end]
        else:
            state['ref_win']  = ref_data.iloc[0:0]  # empty — no overlap
            state['test_win'] = test_rs              # show full test signal

        test_color = '#f87171' if key in rejected else '#60a5fa'
        ax.cla()
        if has_overlap:
            line_ref[0], = ax.plot(state['ref_win'].index, state['ref_win'].values,
                                   color='#aaaaaa', lw=1.2, label='Reference (CH4_ppm)')
        else:
            line_ref[0], = ax.plot([], [], color='#aaaaaa', lw=1.2,
                                   label='Reference (no overlap — MML date)')
        shifted = state['test_win'].index + pd.to_timedelta(lag, unit='s')
        line_test[0], = ax.plot(shifted, state['test_win'].values,
                                color=test_color, lw=1.2, label=f'{test_name} (CH4_ppm)')
        ax.set_xlabel('Time (UTC)')
        ax.set_ylabel('CH4_ppm')
        ax.legend(fontsize=9, loc='upper right')
        ax.set_title(_title())
        fig.autofmt_xdate()

        s_lag.set_val(lag)
        s_scale.set_val(1.0)
        s_voff.set_val(0.0)
        fig.canvas.draw_idle()

    def update(_):
        if state['test_win'] is None:
            return
        shifted = state['test_win'].index + pd.to_timedelta(s_lag.val, unit='s')
        line_test[0].set_xdata(shifted)
        line_test[0].set_ydata(state['test_win'].values * s_scale.val + s_voff.val)
        ax.relim()
        ax.autoscale_view()
        ax.set_title(_title())
        fig.canvas.draw_idle()

    s_lag.on_changed(update)
    s_scale.on_changed(update)
    s_voff.on_changed(update)

    def _advance():
        if state['idx'] < len(test_files) - 1:
            state['idx'] += 1
            load_and_draw(state['idx'])
        else:
            print('  All files reviewed — run the Save cell below.')

    def on_prev(_):
        state['idx'] = max(0, state['idx'] - 1)
        load_and_draw(state['idx'])

    def on_next(_):
        state['idx'] = min(len(test_files) - 1, state['idx'] + 1)
        load_and_draw(state['idx'])

    def on_commit_next(_):
        f   = test_files[state['idx']]
        key = json_key(f)
        lag = float(s_lag.val)
        confirmed[key] = lag
        rejected.discard(key)
        idx = state['idx']
        print(f'  [OK]  [{idx}] {key}  ->  {lag:+.0f}s')
        _advance()

    def on_bad(_):
        f   = test_files[state['idx']]
        key = json_key(f)
        rejected.add(key)
        confirmed.pop(key, None)
        idx = state['idx']
        print(f'  [BAD] [{idx}] {key}')
        _advance()

    btn_prev.on_clicked(on_prev)
    btn_next.on_clicked(on_next)
    btn_commit.on_clicked(on_commit_next)
    btn_bad.on_clicked(on_bad)

    load_and_draw(0)
    # Pin all widget objects to the figure to prevent garbage collection severing callbacks
    fig._slv_widgets = [s_lag, s_scale, s_voff, btn_prev, btn_bad, btn_commit, btn_next]
    plt.show()
    print('Commit & Next = good | Mark Bad & Next = exclude | Next = skip (auto lag saved at commit)')

print('Widget helper loaded.')

## Load Picarro reference

Load all Picarro cleaned files into a single resampled reference series. Used by sections A, B, and C.

In [ ]:
print('Loading Picarro reference (187 files — may take ~30s)...')
picarro_ref = load_all_ref(PICARRO_DIR)
print(f'\nPicarro (CH4_ppm): {len(picarro_ref):,} samples')
print(f'  Range: {picarro_ref.index[0]}  ->  {picarro_ref.index[-1]}')

---
## A — Ultra 460 vs Picarro

Ultra 460 is WYO-only (Feb 3–12). All 44 files should have good Picarro overlap.

In [ ]:
u460_files = sorted(ULTRA460_DIR.glob('*_clean.csv'))
print(f'Ultra 460: {len(u460_files)} files\n')
u460_suggestions = auto_correlate(u460_files, picarro_ref)

### A — Visual review

Step through files with the slider. Use **V-Scale** / **V-Offset** to visually align amplitudes (these are not saved — only the lag is).

In [ ]:
if 'u460_confirmed' not in dir():
    u460_confirmed = {}
if 'u460_rejected' not in dir():
    u460_rejected = set()

make_review_widget(picarro_ref, u460_files, 'Ultra460',
                   u460_suggestions, u460_confirmed, u460_rejected)

### A — Save Ultra 460 offsets

In [ ]:
save_offsets(u460_confirmed, u460_rejected, 'ultra460')

---
## B — Ultra 321 vs Picarro

Ultra 321 WYO dates: Feb 3, Feb 5–12. MML dates: Jan 19–22, Feb 2, Feb 4, Mar 8–10.
MML files have no Picarro overlap — they will auto-suggest **0s** and are expected to show
a flat/empty plot. Commit them at 0 or mark bad if the file looks unusable.

In [ ]:
u321_files = sorted(ULTRA321_DIR.glob('*_clean.csv'))
print(f'Ultra 321: {len(u321_files)} files')
print('(MML files — Feb 4, Mar 8-10 — have no Picarro overlap; will auto-suggest 0s)\n')
u321_suggestions = auto_correlate(u321_files, picarro_ref)

### B — Visual review

In [ ]:
if 'u321_confirmed' not in dir():
    u321_confirmed = {}
if 'u321_rejected' not in dir():
    u321_rejected = set()

make_review_widget(picarro_ref, u321_files, 'Ultra321',
                   u321_suggestions, u321_confirmed, u321_rejected)

### B — Save Ultra 321 offsets

In [ ]:
save_offsets(u321_confirmed, u321_rejected, 'ultra321')

---
## C — Pico 017: WYO dates vs Picarro

WYO files (Feb 5–12 only — Pico017 was not on WYO before Feb 5) are cross-correlated against Picarro.
MML files (Jan 19–22, Feb 4, Mar 8–10) are handled in Section D.

In [ ]:
pico_all_files = sorted(PICO017_DIR.glob('*_clean.csv'))
pico_wyo_files = [f for f in pico_all_files if not is_mml(f)]
pico_mml_files = [f for f in pico_all_files if     is_mml(f)]

print(f'Pico 017 WYO dates : {len(pico_wyo_files)} files (this section — vs Picarro)')
print(f'Pico 017 MML dates : {len(pico_mml_files)} files (Section D — vs Ultra321)\n')
pico_wyo_suggestions = auto_correlate(pico_wyo_files, picarro_ref)

### C — Visual review

In [ ]:
if 'pico_wyo_confirmed' not in dir():
    pico_wyo_confirmed = {}
if 'pico_wyo_rejected' not in dir():
    pico_wyo_rejected = set()

make_review_widget(picarro_ref, pico_wyo_files, 'Pico017',
                   pico_wyo_suggestions, pico_wyo_confirmed, pico_wyo_rejected)

---
## D — Pico 017 (MML dates) + LGR vs Ultra 321

For MML deployments (Feb 4, Mar 8–10), Ultra 321 is the reference — both instruments
were timestamp-corrected to the same Toughbook logger so they are already UTC-aligned;
the cross-correlation measures any residual physical lag (inlet, sampling rate).

The LGR was deployed MML only on Mar 10 and is corrected to the same Toughbook.

In [ ]:
u321_mml_files = [f for f in sorted(ULTRA321_DIR.glob('*_clean.csv')) if is_mml(f)]
print(f'Loading Ultra321 MML reference ({len(u321_mml_files)} files: Feb 4, Mar 8-10)...')
u321_mml_ref = load_ref_from_files(u321_mml_files)
print(f'\nUltra321 MML (CH4_ppm): {len(u321_mml_ref):,} samples')
print(f'  Range: {u321_mml_ref.index[0]}  ->  {u321_mml_ref.index[-1]}')

In [ ]:
print(f'Pico 017 MML dates: {len(pico_mml_files)} files (vs Ultra321)\n')
pico_mml_suggestions = auto_correlate(pico_mml_files, u321_mml_ref)

### D — Pico 017 MML visual review

In [ ]:
if 'pico_mml_confirmed' not in dir():
    pico_mml_confirmed = {}
if 'pico_mml_rejected' not in dir():
    pico_mml_rejected = set()

make_review_widget(u321_mml_ref, pico_mml_files, 'Pico017-MML',
                   pico_mml_suggestions, pico_mml_confirmed, pico_mml_rejected)

In [ ]:
lgr_files = sorted(LGR_DIR.glob('*_clean.csv'))
print(f'LGR: {len(lgr_files)} file (Mar 10 MML — vs Ultra321)\n')
lgr_suggestions = auto_correlate(lgr_files, u321_mml_ref)

### D — LGR visual review

In [ ]:
if 'lgr_confirmed' not in dir():
    lgr_confirmed = {}
if 'lgr_rejected' not in dir():
    lgr_rejected = set()

make_review_widget(u321_mml_ref, lgr_files, 'LGR',
                   lgr_suggestions, lgr_confirmed, lgr_rejected)

### D — Save Pico 017 + LGR offsets

Merges WYO and MML confirmed/rejected dicts before saving.

In [ ]:
pico_confirmed = {**pico_wyo_confirmed, **pico_mml_confirmed}
pico_rejected  = pico_wyo_rejected | pico_mml_rejected

print('=== Pico 017 ===')
save_offsets(pico_confirmed, pico_rejected, 'pico017')

print('\n=== LGR ===')
save_offsets(lgr_confirmed, lgr_rejected, 'lgr')